# Choose an ML GPU Cluster

# LSTM Model —> Sequence-Based Direction Prediction

## Tabular vs. Sequential Models

**Tabular models** look at each row independently. They see one day's features
(RSI, volume, moving averages) and predict that day's label. They have **no memory**
of previous days — row order is irrelevant to them.
- Examples: LogisticRegression, RandomForest, XGBoost, LightGBM (what the AutoML baseline uses).

**Sequential models** read an *ordered window* of days and carry a memory forward,
so they can learn temporal patterns like "prices drifted up for 5 days, then volume
spiked → tomorrow likely down." This notebook uses an **LSTM** (Long Short-Term Memory)
network over a **30-day window**.
- Examples: LSTM, GRU, 1D-CNN, Transformer. Common uses: stock/time-series forecasting,
  text, speech, sensor data.

| | Tabular (AutoML) | Sequential (LSTM) |
|---|---|---|
| Input | one day's features | ordered window of 30 days |
| Memory of past | none | yes (recurrent state) |
| Order matters? | no | yes |
| Examples | XGBoost, RandomForest | LSTM, GRU, Transformer |

**One-line difference:** the tabular model answers *"given today's numbers, up or down?"*;
the LSTM answers *"given the last 30 days in order, up or down tomorrow?"*

## Point of This Notebook

The AutoML baseline notebook established a fast, trustworthy reference score
(~52% accuracy is realistic and genuinely good for markets).

This notebook tests whether a custom **sequence** model that exploits day-to-day
ordering can **beat that baseline**. Both notebooks predict the same target on the
same data, so the only real difference is *independent rows* vs *30-day sequences*.

- If the LSTM beats AutoML → temporal patterns add real signal, and the extra
  complexity is justified.
- If it doesn't → that's also a valid result: ordering isn't adding predictive value
  here, learned cheaply and with evidence rather than assumption.


## Why We Judge Models by F1, Not Accuracy

### What is F1 score?
F1 is the harmonic mean of **precision** and **recall**, computed per class:

- **Precision** = of all the times the model said "Up", how many were actually Up?
- **Recall**    = of all the actual Up days, how many did the model catch?
- **F1**        = 2 × (precision × recall) / (precision + recall)

A high F1 requires the model to be good at **both** — it can't cheat by ignoring a class. Precision catches this issue. You made 3650 "Up" calls, but only 1863 were truly Up. → precision = 1863 / 3650 = 0.51, NOT perfect.

### Why high accuracy can be misleading here
Our test set is imbalanced: **1863 Up vs 1787 Down**.
So a model that **always predicts "Up"** — with zero skill — scores:

    1863 / 3650 = 0.5104 accuracy

That means ~51% accuracy is the "do nothing" floor, not a real achievement.

### Why tiny_lowlr (0.514 acc) and high_reg (0.514 acc) are NOT good
- Both have **high accuracy but low F1** (tiny_lowlr F1=0.41, high_reg F1=0.38).
- That gap is the tell: they collapsed to predicting mostly one class.
- tiny_lowlr even stopped at **epoch 1** — it basically never learned.
- Their accuracy ≈ the trivial "always predict Up" baseline (0.5104). No real skill.

### Why F1 wins
- **Accuracy can be faked** by predicting the majority class.
- **F1 cannot** — it punishes a model that ignores one class.
- A useful trading model must call **both Up and Down**; F1 rewards exactly that.


## GPU Check

In [0]:
import torch, torchvision
print("Torch:", torch.__version__)
print("Cuda available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load and Prepare Sequence Data

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql.functions import col

# Load Gold features
gold = spark.table("raghavan_trading_signals.gold.daily_features")

# Same metadata columns as AutoML checkpoint
metadata_cols = [
    "symbol", "trade_date", "open", "high", "low", "close", "adj_close",
    "volume", "dividends", "stock_splits", "prev_close",
    "bronze_ingested_at", "bronze_source_file",
    "next_day_return",
]

target_col = "next_day_direction"

feature_cols = [
    c for c in gold.columns
    if c not in metadata_cols and c != target_col
]

print(f"Using {len(feature_cols)} features")

# Convert to Pandas (LSTM training needs NumPy arrays)
# For 50 stocks × ~940 days, this fits comfortably in memory
pdf = (
    gold
    .select("symbol", "trade_date", *feature_cols, target_col)
    .orderBy("symbol", "trade_date")
    .toPandas()
)

print(f"Total rows: {len(pdf):,}")

## Create Sequences

In [0]:
# LSTM input shape: (batch_size, sequence_length, num_features)
# We create sliding windows of 30 days → predict day 31's direction.

SEQUENCE_LENGTH = 30

def create_sequences(df, feature_columns, target_column, seq_len):
    """
    For each stock, slide a window of `seq_len` days across the time series.
    Each window becomes one training sample.

    Example with seq_len=3:
      Day 1, Day 2, Day 3 → predict Day 4
      Day 2, Day 3, Day 4 → predict Day 5
      ...
    """
    X_all, y_all, meta_all = [], [], []

    for symbol, group in df.groupby("symbol"):
        group = group.sort_values("trade_date").reset_index(drop=True)
        features = group[feature_columns].values.astype(np.float32)
        targets = group[target_column].values.astype(np.int64)
        dates = group["trade_date"].values

        for i in range(seq_len, len(group)):
            X_all.append(features[i - seq_len : i])  # shape: (seq_len, num_features)
            y_all.append(targets[i])                   # scalar: 0 or 1
            meta_all.append({"symbol": symbol, "date": dates[i]})

    return np.array(X_all), np.array(y_all), meta_all

X, y, meta = create_sequences(pdf, feature_cols, target_col, SEQUENCE_LENGTH)

print(f"Sequences created: {X.shape[0]:,}")
print(f"Sequence shape: {X.shape} → (samples, timesteps, features)")
print(f"Target distribution: {np.bincount(y)} → [down, up]")

### Interpretation:

#### Sequences created
You built 65,050 training examples, where each example is a 30-day window (sliding window over your data).

#### Sequence shape: (65050, 30, 41)
the input tensor for the LSTM:
- 65050 = number of samples (windows)
- 30 = timesteps (each window is 30 trading days, in order)
- 41 = features per day (RSI, volume, moving averages, etc.)

So each sample is a 30×41 grid, and the LSTM reads the 30 days one at a time.
#### Target distribution: [31230 33820] → [down, up]

The labels: 31,230 windows are followed by a "down" day, 33,820 by an "up" day. That's roughly 48% / 52%, i.e. well balanced and good, because you won't need heavy class-weighting and accuracy won't be misleading.

## Time-Based Train/Test Split



In [0]:
split_date = pd.Timestamp("2025-01-01")

# Each sequence's date (the date its label belongs to)
dates = pd.to_datetime([m["date"] for m in meta])

train_mask = dates < split_date
test_mask  = dates >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {X_train.shape[0]:,} sequences")
print(f"Test:  {X_test.shape[0]:,} sequences")

## Normalize Features

In [0]:
# Neural networks work best when features are on similar scales.
# Fit the scaler on TRAINING data only (never peek at test data).

from sklearn.preprocessing import StandardScaler

num_features = X_train.shape[2]

# Reshape to 2D for the scaler, then reshape back
X_train_flat = X_train.reshape(-1, num_features)
scaler = StandardScaler()
scaler.fit(X_train_flat)

X_train_scaled = scaler.transform(X_train_flat).reshape(X_train.shape)
X_test_scaled = scaler.transform(
    X_test.reshape(-1, num_features)
).reshape(X_test.shape)

# Replace any NaN/Inf that might have slipped through
X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

print("Normalization complete.")
print(f"Train feature mean: {X_train_scaled.mean():.4f} (should be ~0)")
print(f"Train feature std:  {X_train_scaled.std():.4f} (should be ~1)")

## Create PyTorch DataLoaders

In [0]:
import torch
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 256

# Convert to PyTorch tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_scaled),
    torch.LongTensor(y_train)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_scaled),
    torch.LongTensor(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Define the LSTM Model

In [0]:
import torch.nn as nn

class TradingLSTM(nn.Module):
    """
    LSTM for binary classification of next-day stock direction.

    Architecture:
      Input (seq_len, num_features)
        → LSTM layers (capture temporal patterns)
        → Dropout (prevent overfitting)
        → Fully connected layer
        → Sigmoid (output probability of "up")
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(TradingLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # Input shape: (batch, seq, features)
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False,
        )

        self.dropout = nn.Dropout(dropout)

        # Take the LSTM's final hidden state → binary prediction
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)  # 2 classes: down (0), up (1)

    def forward(self, x):
        # x shape: (batch_size, seq_len, num_features)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Use the last hidden state of the last layer
        last_hidden = h_n[-1]  # shape: (batch_size, hidden_size)

        out = self.dropout(last_hidden)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)     # shape: (batch_size, 2)

        return out

# 1) Manual Experimentation with different parameters

In [0]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import mlflow
from sklearn.metrics import accuracy_score, f1_score

# Same experiment as before so all runs are comparable in one place
experiment_name = "/Users/{}/raghavan-trading-signals-lstm-tuning".format(
    spark.sql("SELECT current_user()").collect()[0][0]
)
mlflow.set_experiment(experiment_name)

# Build loaders once (data is identical across experiments)
train_ds = TensorDataset(
    torch.FloatTensor(X_train_scaled), torch.LongTensor(y_train)
)
test_ds = TensorDataset(
    torch.FloatTensor(X_test_scaled), torch.LongTensor(y_test)
)


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_experiment(cfg):
    """Train one config, evaluate on the held-out test set, log to MLflow.

    Returns a dict of summary metrics for the comparison table.
    """
    set_seed(42)  # same init/shuffling so differences come from the config

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False)

    model = TradingLSTM(
        input_size=len(feature_cols),
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    ).to(device)
    model.lstm.flatten_parameters()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_f1, best_epoch, epochs_no_improve = 0.0, 0, 0
    ckpt = f"/tmp/exp_{cfg['name']}.pt"

    with mlflow.start_run(run_name=cfg["name"]) as run:
        mlflow.log_params(cfg)
        mlflow.log_param("num_features", len(feature_cols))

        for epoch in range(cfg["max_epochs"]):
            # --- train ---
            model.train()
            running_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                out = model(xb)
                loss = criterion(out, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                running_loss += loss.item() * xb.size(0)
            train_loss = running_loss / len(train_ds)

            # --- eval ---
            model.eval()
            preds, labels = [], []
            with torch.no_grad():
                for xb, yb in test_loader:
                    out = model(xb.to(device))
                    preds.extend(torch.argmax(out, 1).cpu().numpy())
                    labels.extend(yb.numpy())
            test_acc = accuracy_score(labels, preds)
            test_f1 = f1_score(labels, preds, average="weighted")

            scheduler.step(train_loss)
            mlflow.log_metrics(
                {"train_loss": train_loss, "test_accuracy": test_acc, "test_f1": test_f1},
                step=epoch,
            )

            # --- early stopping on test F1 ---
            if test_f1 > best_f1:
                best_f1, best_epoch = test_f1, epoch
                best_acc = test_acc
                epochs_no_improve = 0
                torch.save(model.state_dict(), ckpt)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= cfg["patience"]:
                    print(f"  [{cfg['name']}] early stop at epoch {epoch+1}")
                    break

        mlflow.log_metrics(
            {"best_test_f1": best_f1, "best_test_accuracy": best_acc, "best_epoch": best_epoch}
        )

    print(
        f"DONE {cfg['name']:18s} | best F1 {best_f1:.4f} | "
        f"best acc {best_acc:.4f} | epoch {best_epoch+1}"
    )
    return {
        "name": cfg["name"],
        "best_test_f1": round(best_f1, 4),
        "best_test_accuracy": round(best_acc, 4),
        "best_epoch": best_epoch + 1,
    }

In [0]:
COMMON = {"batch_size": 64, "max_epochs": 40, "patience": 7}

experiments = [
    # 1. Baseline — your current settings (reference point)
    {"name": "baseline", "hidden_size": 128, "num_layers": 2,
     "dropout": 0.3, "lr": 1e-3, "weight_decay": 1e-5, **COMMON},

    # 2. Stronger regularization (fight overfitting)
    {"name": "high_reg", "hidden_size": 128, "num_layers": 2,
     "dropout": 0.5, "lr": 1e-3, "weight_decay": 1e-4, **COMMON},

    # 3. Smaller model (less capacity to memorize)
    {"name": "small", "hidden_size": 64, "num_layers": 1,
     "dropout": 0.3, "lr": 1e-3, "weight_decay": 1e-5, **COMMON},

    # 4. Small + strong regularization (combined)
    {"name": "small_high_reg", "hidden_size": 64, "num_layers": 1,
     "dropout": 0.5, "lr": 1e-3, "weight_decay": 1e-4, **COMMON},

    # 5. Tiny + low LR (most conservative)
    {"name": "tiny_lowlr", "hidden_size": 32, "num_layers": 1,
     "dropout": 0.5, "lr": 5e-4, "weight_decay": 1e-3, **COMMON},
]

In [0]:
results = []
for cfg in experiments:
    print(f"\n=== Running: {cfg['name']} ===")
    results.append(run_experiment(cfg))


## LSTM Manual Experiment Result

| Name           | Hidden | Layers | Dropout | LR    | Weight Decay | Best F1 | Best Acc | Best Epoch | What changed vs baseline                  |
|----------------|:------:|:------:|:-------:|:-----:|:------------:|:-------:|:--------:|:----------:|-------------------------------------------|
| baseline       | 128    | 2      | 0.3     | 1e-3  | 1e-5         | 0.4926  | 0.4940   | 30         | Reference config                          |
| small          | 64     | 1      | 0.3     | 1e-3  | 1e-5         | 0.4663  | 0.4671   | 8          | ½ capacity, fewer layers                  |
| small_high_reg | 64     | 1      | 0.5     | 1e-3  | 1e-4         | 0.4076  | 0.4932   | 8          | Smaller **and** stronger regularization   |
| tiny_lowlr     | 32     | 1      | 0.5     | 5e-4  | 1e-3         | 0.4057  | 0.5142   | 1          | Smallest, lowest LR, strongest WD         |
| high_reg       | 128    | 2      | 0.5     | 1e-3  | 1e-4         | 0.3756  | 0.5140   | 4          | Same size, heavy dropout + weight decay   |


### What each knob does
- **hidden_size / num_layers** — model *capacity*. Bigger = can learn more complex patterns, but also memorizes noise (overfits) more easily. `baseline` (128×2) is the biggest; `tiny_lowlr` (32×1) is the smallest.
- **dropout** — randomly zeroes activations during training to prevent co-adaptation. 0.3 = mild, 0.5 = aggressive regularization.
- **weight_decay** — L2 penalty that shrinks weights toward zero. 1e-5 = barely on, 1e-3 = strong. Fights overfitting.
- **lr** — step size for Adam. Lower (5e-4) = slower, more conservative learning.

### How to read the results
- **F1 is the metric that matters here, not accuracy.** Notice `high_reg` and `tiny_lowlr` have the *highest accuracy* (~0.51) but the *lowest F1* (~0.38–0.41). That's the classic signature of a model that **collapsed to predicting the majority class** — heavy regularization + low LR killed its ability to discriminate, so it just predicts the most common label and rides class imbalance to a deceptively high accuracy. F1 (weighted) exposes that the minority classes are being ignored.
- **baseline wins on F1 (0.4926).** Reducing capacity (`small`) or piling on regularization (`*_high_reg`, `tiny_lowlr`) all *hurt* F1. So at this data size the model is **not** overfitting badly enough to justify shrinking it — the regularization experiments over-corrected.
- **best_epoch tells you about convergence.** baseline kept improving until epoch 30; the heavily-regularized runs peaked at epoch 1–8 then early-stopped, i.e. they plateaued almost immediately (another sign they collapsed rather than learned).

# 2) Training Loop with Baseline model but with higher epochs

In [0]:
import mlflow
import mlflow.pytorch
from sklearn.metrics import accuracy_score, f1_score, classification_report

# === HYPERPARAMETERS ===
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
WEIGHT_DECAY = 1e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Initialize model
model = TradingLSTM(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler: reduce LR when loss plateaus.
# Log LR changes manually in the training loop if you want the printout.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model architecture:\n{model}")

In [0]:
# === TRAINING LOOP ===

experiment_name = "/Users/{}/raghavan-trading-signals-lstm-tuning".format(
    spark.sql("SELECT current_user()").collect()[0][0]
)
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="lstm_v1") as run:

    # Log all hyperparameters
    mlflow.log_params({
        "model_type": "LSTM",
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "sequence_length": SEQUENCE_LENGTH,
        "batch_size": BATCH_SIZE,
        "num_features": len(feature_cols),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "device": str(device),
    })

    best_test_f1 = 0
    train_losses = []

    # Ensure the model is on the correct device, then rebuild the LSTM's cuDNN
    # weight buffer. An nn.LSTM caches its parameters in `_flat_weights`; if the
    # module was ever moved between devices in this Python session, that buffer
    # can desync from the real parameters and cuDNN will still see CPU weights
    # (RuntimeError: input ... at cuda:0 and parameter tensor at cpu) even after
    # `.to(device)`. flatten_parameters() rebuilds it from the current params.
    model.to(device)
    model.lstm.flatten_parameters()

    for epoch in range(NUM_EPOCHS):
        # --- Training phase ---
        model.train()
        epoch_loss = 0
        epoch_correct = 0
        epoch_total = 0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()

            # Gradient clipping to prevent exploding gradients in LSTMs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            epoch_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            epoch_correct += (predicted == batch_y).sum().item()
            epoch_total += batch_y.size(0)

        avg_train_loss = epoch_loss / epoch_total
        train_accuracy = epoch_correct / epoch_total
        train_losses.append(avg_train_loss)

        # --- Evaluation phase ---
        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                outputs = model(batch_X)
                _, predicted = torch.max(outputs, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(batch_y.numpy())

        test_accuracy = accuracy_score(all_labels, all_preds)
        test_f1 = f1_score(all_labels, all_preds, average="weighted")

        # Step the scheduler
        scheduler.step(avg_train_loss)

        # Log metrics to MLflow at each epoch
        mlflow.log_metrics({
            "train_loss": avg_train_loss,
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "test_f1": test_f1,
        }, step=epoch)

        # Track best model
        if test_f1 > best_test_f1:
            best_test_f1 = test_f1
            best_epoch = epoch
            # Save the best model checkpoint
            torch.save(model.state_dict(), "/tmp/best_lstm_model.pt")

        # Print progress every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Train Acc: {train_accuracy:.4f} | "
                f"Test Acc: {test_accuracy:.4f} | "
                f"Test F1: {test_f1:.4f}"
            )

    # --- Final evaluation ---
    # Reload the best model
    model.load_state_dict(torch.load("/tmp/best_lstm_model.pt"))
    model.eval()

    final_preds = []
    final_probs = []
    final_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            final_preds.extend(predicted.cpu().numpy())
            final_probs.extend(probs[:, 1].cpu().numpy())  # probability of "up"
            final_labels.extend(batch_y.numpy())

    final_accuracy = accuracy_score(final_labels, final_preds)
    final_f1 = f1_score(final_labels, final_preds, average="weighted")

    print(f"\n{'='*60}")
    print(f"Best model from epoch {best_epoch + 1}")
    print(f"Test Accuracy: {final_accuracy:.4f}")
    print(f"Test F1:       {final_f1:.4f}")
    print(f"\n{classification_report(final_labels, final_preds, target_names=['Down', 'Up'])}")

    # Log final metrics
    mlflow.log_metrics({
        "best_test_accuracy": final_accuracy,
        "best_test_f1": final_f1,
        "best_epoch": best_epoch,
    })

    # Log the model artifact to MLflow.
    # Unity Catalog REQUIRES a model signature (input + output schema), so we
    # infer one from a small real example shaped like one batch: (n, 30, 41).
    #
    # IMPORTANT: do NOT do `model.to("cpu")` here. nn.Module.to() is in-place,
    # so it would move your live training model to CPU; if anything below fails
    # or the cell is re-run, the LSTM is left stranded on CPU with desynced
    # internal weights and a later `model.to(device)` won't reliably fix it.
    # Instead, keep the model on its device and just move the tiny example.
    from mlflow.models.signature import infer_signature

    model.eval()
    example_input = X_test_scaled[:2].astype("float32")  # (2, 30, 41)
    with torch.no_grad():
        example_output = (
            model(torch.from_numpy(example_input).to(device)).cpu().numpy()
        )  # (2, 2) class logits

    signature = infer_signature(example_input, example_output)

    # log_model saves the state dict, so logging an on-GPU model is fine.
    mlflow.pytorch.log_model(
        model,
        artifact_path="model",
        registered_model_name="raghavan_trading_signals.ml.lstm_signal_model",
        signature=signature,
        input_example=example_input,
    )

    # Log the scaler as an artifact (needed for inference)
    import joblib
    joblib.dump(scaler, "/tmp/feature_scaler.pkl")
    mlflow.log_artifact("/tmp/feature_scaler.pkl")

    # Log feature column names (needed for inference)
    with open("/tmp/feature_cols.txt", "w") as f:
        f.write("\n".join(feature_cols))
    mlflow.log_artifact("/tmp/feature_cols.txt")

    lstm_run_id = run.info.run_id
    print(f"\nMLflow run ID: {lstm_run_id}")


## Results for Baseline model but with higher epochs

- Best Model: Epoch 45
- **Test Accuracy:** 0.4932  
- **Test F1:** 0.4932
| Class        | Precision | Recall | F1-Score | Support |
|--------------|:---------:|:------:|:--------:|:-------:|
| Down         | 0.48      | 0.49   | 0.49     | 1787    |
| Up           | 0.50      | 0.49   | 0.50     | 1863    |
| **Accuracy** |           |        | **0.49** | 3650    |
| Macro Avg    | 0.49      | 0.49   | 0.49     | 3650    |
| Weighted Avg | 0.49      | 0.49   | 0.49     | 3650    |

# 3) LSTM Hyperparameter Tuning using Random Search

In [0]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", device)

In [0]:
# === CONFIG — edit here, everything below reads from these ===
CATALOG = "raghavan_trading_signals"
SOURCE_TABLE = f"{CATALOG}.gold.daily_features"
TARGET_COL = "next_day_direction"

SEQUENCE_LENGTH = 30

# Time-based 3-way split (oldest -> train, middle -> val, newest -> test).
# We tune/early-stop on VAL and touch TEST only once at the very end.
VAL_SPLIT_DATE = "2024-10-01"   # train  = dates <  VAL_SPLIT_DATE
TEST_SPLIT_DATE = "2025-01-01"  # val    = [VAL_SPLIT_DATE, TEST_SPLIT_DATE); test = >=

N_TRIALS = 15            # how many hyperparameter configs to try
MAX_EPOCHS = 40
EARLY_STOP_PATIENCE = 7
SEED = 42

REGISTERED_MODEL_NAME = f"{CATALOG}.ml.lstm_signal_model"

In [0]:
from pyspark.sql.functions import col

gold = spark.table(SOURCE_TABLE)

metadata_cols = [
    "symbol", "trade_date", "open", "high", "low", "close", "adj_close",
    "volume", "dividends", "stock_splits", "prev_close",
    "bronze_ingested_at", "bronze_source_file",
    "next_day_return",
]
feature_cols = [c for c in gold.columns if c not in metadata_cols and c != TARGET_COL]
print(f"Using {len(feature_cols)} features")

# 50 stocks x ~940 days fits comfortably in driver memory as Pandas.
pdf = (
    gold.select("symbol", "trade_date", *feature_cols, TARGET_COL)
    .orderBy("symbol", "trade_date")
    .toPandas()
)
print(f"Total rows: {len(pdf):,}")

In [0]:
# Sliding windows of SEQUENCE_LENGTH days -> predict the next day.
def create_sequences(df, feature_columns, target_column, seq_len):
    X_all, y_all, meta_all = [], [], []
    for symbol, group in df.groupby("symbol"):
        group = group.sort_values("trade_date").reset_index(drop=True)
        features = group[feature_columns].values.astype(np.float32)
        targets = group[target_column].values.astype(np.int64)
        dates = group["trade_date"].values
        for i in range(seq_len, len(group)):
            X_all.append(features[i - seq_len : i])
            y_all.append(targets[i])
            meta_all.append({"symbol": symbol, "date": dates[i]})
    return np.array(X_all), np.array(y_all), meta_all

X, y, meta = create_sequences(pdf, feature_cols, TARGET_COL, SEQUENCE_LENGTH)
print(f"Sequences: {X.shape[0]:,}  shape={X.shape}  (samples, timesteps, features)")
print(f"Target distribution: {np.bincount(y)}  [down, up]")

In [0]:
# 3-way time-ordered split. No shuffling across time -> no future leakage.
val_start = pd.Timestamp(VAL_SPLIT_DATE)
test_start = pd.Timestamp(TEST_SPLIT_DATE)

dates = pd.to_datetime([m["date"] for m in meta])
train_mask = np.asarray(dates < val_start)
val_mask = np.asarray((dates >= val_start) & (dates < test_start))
test_mask = np.asarray(dates >= test_start)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"Train dist {np.bincount(y_train)} | Val dist {np.bincount(y_val)} | Test dist {np.bincount(y_test)}")

In [0]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler on TRAIN only, apply to all three splits.
num_features = X_train.shape[2]
scaler = StandardScaler().fit(X_train.reshape(-1, num_features))

def scale(a):
    out = scaler.transform(a.reshape(-1, num_features)).reshape(a.shape)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

X_train_scaled = scale(X_train)
X_val_scaled = scale(X_val)
X_test_scaled = scale(X_test)
print("Scaled. train mean ~", round(float(X_train_scaled.mean()), 4),
      "| std ~", round(float(X_train_scaled.std()), 4))

In [0]:
train_ds = TensorDataset(torch.FloatTensor(X_train_scaled), torch.LongTensor(y_train))
val_ds = TensorDataset(torch.FloatTensor(X_val_scaled), torch.LongTensor(y_val))
test_ds = TensorDataset(torch.FloatTensor(X_test_scaled), torch.LongTensor(y_test))

# Class weights from TRAIN labels. Without this, heavy regularization makes
# the model collapse to predicting the majority class (high accuracy, low F1).
counts = np.bincount(y_train, minlength=2).astype(np.float64)
class_weights = torch.tensor(counts.sum() / (2.0 * counts), dtype=torch.float32).to(device)
print("Train class counts:", counts, "-> weights:", class_weights.cpu().numpy())

## Model Training Framework

In [0]:
class TradingLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)        # last hidden state of last layer
        out = self.dropout(h_n[-1])
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        return self.fc2(out)

In [0]:
import mlflow
import mlflow.pytorch
from sklearn.metrics import accuracy_score, f1_score

experiment_name = "/Users/{}/raghavan-trading-signals-lstm-tuning".format(
    spark.sql("SELECT current_user()").collect()[0][0]
)
mlflow.set_experiment(experiment_name)

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for xb, yb in loader:
        out = model(xb.to(device))
        preds.extend(torch.argmax(out, 1).cpu().numpy())
        labels.extend(yb.numpy())
    return accuracy_score(labels, preds), f1_score(labels, preds, average="weighted")

def train_and_eval(cfg, nested=False):
    """Train one config; early-stop & SELECT on validation F1.
    Logs an MLflow run and returns a summary dict + best checkpoint path."""
    set_seed(SEED)  # same init/shuffling so differences come from the config
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False)

    model = TradingLSTM(
        input_size=len(feature_cols),
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    ).to(device)
    model.lstm.flatten_parameters()

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_val_f1, best_val_acc, best_epoch, no_improve = 0.0, 0.0, 0, 0
    ckpt = f"/tmp/lstm_{cfg['name']}.pt"

    with mlflow.start_run(run_name=cfg["name"], nested=nested):
        mlflow.log_params(cfg)
        mlflow.log_param("num_features", len(feature_cols))
        for epoch in range(cfg["max_epochs"]):
            model.train()
            running = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                running += loss.item() * xb.size(0)
            train_loss = running / len(train_ds)

            val_acc, val_f1 = evaluate(model, val_loader)
            scheduler.step(train_loss)
            mlflow.log_metrics(
                {"train_loss": train_loss, "val_accuracy": val_acc, "val_f1": val_f1},
                step=epoch,
            )

            if val_f1 > best_val_f1:
                best_val_f1, best_val_acc, best_epoch, no_improve = val_f1, val_acc, epoch, 0
                torch.save(model.state_dict(), ckpt)
            else:
                no_improve += 1
                if no_improve >= cfg["patience"]:
                    break
        mlflow.log_metrics(
            {"best_val_f1": best_val_f1, "best_val_accuracy": best_val_acc, "best_epoch": best_epoch}
        )
    print(f"DONE {cfg['name']:14s} | val F1 {best_val_f1:.4f} | val acc {best_val_acc:.4f} | epoch {best_epoch+1}")
    return {"name": cfg["name"], "val_f1": round(best_val_f1, 4), "val_acc": round(best_val_acc, 4),
            "best_epoch": best_epoch + 1, "ckpt": ckpt, "cfg": cfg}

In [0]:
import itertools, random

SEARCH_SPACE = {
    "hidden_size":  [32, 64, 128, 256],
    "num_layers":   [1, 2, 3],
    "dropout":      [0.2, 0.3, 0.4, 0.5],
    "lr":           [5e-4, 1e-3, 2e-3],
    "weight_decay": [1e-5, 1e-4, 1e-3],
    "batch_size":   [64, 128, 256],
}

# Random search: sample N_TRIALS combos instead of all
# 4*3*4*3*3*3 = 1296 (random search beats grid for the same budget).
random.seed(SEED)
keys = list(SEARCH_SPACE)
all_combos = list(itertools.product(*SEARCH_SPACE.values()))
sampled = random.sample(all_combos, min(N_TRIALS, len(all_combos)))

results = []
with mlflow.start_run(run_name="random_search_parent"):
    for i, combo in enumerate(sampled):
        cfg = dict(zip(keys, combo))
        cfg.update({"name": f"rand_{i:02d}", "max_epochs": MAX_EPOCHS, "patience": EARLY_STOP_PATIENCE})
        print(f"\n=== Trial {i+1}/{len(sampled)}: {cfg['name']} === {combo}")
        results.append(train_and_eval(cfg, nested=True))

In [0]:
res_df = pd.DataFrame([
    {"name": r["name"], "val_f1": r["val_f1"], "val_acc": r["val_acc"], "best_epoch": r["best_epoch"],
     **{k: r["cfg"][k] for k in ["hidden_size", "num_layers", "dropout", "lr", "weight_decay", "batch_size"]}}
    for r in results
]).sort_values("val_f1", ascending=False).reset_index(drop=True)
display(res_df)

best = max(results, key=lambda r: r["val_f1"])
print("BEST config (by validation F1):", best["cfg"])

In [0]:
from sklearn.metrics import classification_report

# Rebuild the best model and load the checkpoint that won on validation F1.
best_cfg = best["cfg"]
best_model = TradingLSTM(
    input_size=len(feature_cols),
    hidden_size=best_cfg["hidden_size"],
    num_layers=best_cfg["num_layers"],
    dropout=best_cfg["dropout"],
).to(device)
best_model.load_state_dict(torch.load(best["ckpt"]))
best_model.lstm.flatten_parameters()

test_loader = DataLoader(test_ds, batch_size=best_cfg["batch_size"], shuffle=False)
test_acc, test_f1 = evaluate(best_model, test_loader)
print(f"=== HELD-OUT TEST (best config: {best_cfg['name']}) ===")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test F1:       {test_f1:.4f}\n")

best_model.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.extend(torch.argmax(best_model(xb.to(device)), 1).cpu().numpy())
        labels.extend(yb.numpy())
print(classification_report(labels, preds, target_names=["Down", "Up"]))

In [0]:
from mlflow.models.signature import infer_signature
import joblib

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name=f"best_{best_cfg['name']}_final"):
    mlflow.log_params(best_cfg)
    mlflow.log_metrics({"test_accuracy": test_acc, "test_f1": test_f1})

    example_input = X_test_scaled[:2].astype("float32")  # (2, 30, num_features)
    with torch.no_grad():
        example_output = best_model(torch.from_numpy(example_input).to(device)).cpu().numpy()
    signature = infer_signature(example_input, example_output)

    mlflow.pytorch.log_model(
        best_model,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME,
        signature=signature,
        input_example=example_input,
    )

    joblib.dump(scaler, "/tmp/feature_scaler.pkl")
    mlflow.log_artifact("/tmp/feature_scaler.pkl")
    with open("/tmp/feature_cols.txt", "w") as f:
        f.write("\n".join(feature_cols))
    mlflow.log_artifact("/tmp/feature_cols.txt")
    print("Logged & registered:", REGISTERED_MODEL_NAME)

## Random Search (Conclusion)

15 randomly-sampled configs, each trained with a **3-way time split** (train 58,200 / val 3,200 / test 3,650), **class-weighted** loss, early-stopped and **selected on validation F1**. The test set is touched exactly once, at the very end, so its number is an honest estimate.

**Winner -> `rand_07`:** hidden=256, layers=2, dropout=0.3, lr=1e-3, weight_decay=1e-5, batch=64

| Class | Precision | Recall | F1 | Support |
|---|:--:|:--:|:--:|:--:|
| Down | 0.49 | 0.39 | 0.43 | 1787 |
| Up | 0.51 | 0.62 | 0.56 | 1863 |
| **Weighted avg** | 0.50 | 0.51 | **0.50** | 3650 |

**What changed vs the earlier hand-picked experiments:** adding a dedicated validation split *and* class weights fixed the majority-class collapse seen before. The model now calls **both** directions (Down recall 0.39, Up recall 0.62) instead of always shouting "Up" — which is exactly why its F1 (0.50) clears AutoML's (0.45). The ~0.51 accuracy is still the honest market floor: this is a **better-behaved** model, not a more **skillful** one. Optuna (next section) gets one more shot at finding signal before we conclude.

# 4) LSTM Hyperparameter Tuning using Optuna

In [0]:
# Optuna is not pre-installed on the ML runtime. Install + restart Python so
# the import is available. "Run All" continues through the restart.
%pip install optuna
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", device)

In [0]:
# === CONFIG — edit here, everything below reads from these ===
CATALOG = "raghavan_trading_signals"
SOURCE_TABLE = f"{CATALOG}.gold.daily_features"
TARGET_COL = "next_day_direction"

SEQUENCE_LENGTH = 30

# Time-based 3-way split (oldest -> train, middle -> val, newest -> test).
# We tune/early-stop on VAL and touch TEST only once at the very end.
VAL_SPLIT_DATE = "2024-10-01"   # train  = dates <  VAL_SPLIT_DATE
TEST_SPLIT_DATE = "2025-01-01"  # val    = [VAL_SPLIT_DATE, TEST_SPLIT_DATE); test = >=

N_TRIALS = 15            # how many hyperparameter configs to try
MAX_EPOCHS = 40
EARLY_STOP_PATIENCE = 7
SEED = 42

REGISTERED_MODEL_NAME = f"{CATALOG}.ml.lstm_signal_model"

In [0]:
from pyspark.sql.functions import col

gold = spark.table(SOURCE_TABLE)

metadata_cols = [
    "symbol", "trade_date", "open", "high", "low", "close", "adj_close",
    "volume", "dividends", "stock_splits", "prev_close",
    "bronze_ingested_at", "bronze_source_file",
    "next_day_return",
]
feature_cols = [c for c in gold.columns if c not in metadata_cols and c != TARGET_COL]
print(f"Using {len(feature_cols)} features")

# 50 stocks x ~940 days fits comfortably in driver memory as Pandas.
pdf = (
    gold.select("symbol", "trade_date", *feature_cols, TARGET_COL)
    .orderBy("symbol", "trade_date")
    .toPandas()
)
print(f"Total rows: {len(pdf):,}")

In [0]:
# Sliding windows of SEQUENCE_LENGTH days -> predict the next day.
def create_sequences(df, feature_columns, target_column, seq_len):
    X_all, y_all, meta_all = [], [], []
    for symbol, group in df.groupby("symbol"):
        group = group.sort_values("trade_date").reset_index(drop=True)
        features = group[feature_columns].values.astype(np.float32)
        targets = group[target_column].values.astype(np.int64)
        dates = group["trade_date"].values
        for i in range(seq_len, len(group)):
            X_all.append(features[i - seq_len : i])
            y_all.append(targets[i])
            meta_all.append({"symbol": symbol, "date": dates[i]})
    return np.array(X_all), np.array(y_all), meta_all

X, y, meta = create_sequences(pdf, feature_cols, TARGET_COL, SEQUENCE_LENGTH)
print(f"Sequences: {X.shape[0]:,}  shape={X.shape}  (samples, timesteps, features)")
print(f"Target distribution: {np.bincount(y)}  [down, up]")

In [0]:
# 3-way time-ordered split. No shuffling across time -> no future leakage.
val_start = pd.Timestamp(VAL_SPLIT_DATE)
test_start = pd.Timestamp(TEST_SPLIT_DATE)

dates = pd.to_datetime([m["date"] for m in meta])
train_mask = np.asarray(dates < val_start)
val_mask = np.asarray((dates >= val_start) & (dates < test_start))
test_mask = np.asarray(dates >= test_start)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"Train dist {np.bincount(y_train)} | Val dist {np.bincount(y_val)} | Test dist {np.bincount(y_test)}")

In [0]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler on TRAIN only, apply to all three splits.
num_features = X_train.shape[2]
scaler = StandardScaler().fit(X_train.reshape(-1, num_features))

def scale(a):
    out = scaler.transform(a.reshape(-1, num_features)).reshape(a.shape)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

X_train_scaled = scale(X_train)
X_val_scaled = scale(X_val)
X_test_scaled = scale(X_test)
print("Scaled. train mean ~", round(float(X_train_scaled.mean()), 4),
      "| std ~", round(float(X_train_scaled.std()), 4))

In [0]:
train_ds = TensorDataset(torch.FloatTensor(X_train_scaled), torch.LongTensor(y_train))
val_ds = TensorDataset(torch.FloatTensor(X_val_scaled), torch.LongTensor(y_val))
test_ds = TensorDataset(torch.FloatTensor(X_test_scaled), torch.LongTensor(y_test))

# Class weights from TRAIN labels. Without this, heavy regularization makes
# the model collapse to predicting the majority class (high accuracy, low F1).
counts = np.bincount(y_train, minlength=2).astype(np.float64)
class_weights = torch.tensor(counts.sum() / (2.0 * counts), dtype=torch.float32).to(device)
print("Train class counts:", counts, "-> weights:", class_weights.cpu().numpy())

In [0]:
class TradingLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)        # last hidden state of last layer
        out = self.dropout(h_n[-1])
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        return self.fc2(out)

In [0]:
import mlflow
import mlflow.pytorch
from sklearn.metrics import accuracy_score, f1_score

experiment_name = "/Users/{}/raghavan-trading-signals-lstm-tuning".format(
    spark.sql("SELECT current_user()").collect()[0][0]
)
mlflow.set_experiment(experiment_name)

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for xb, yb in loader:
        out = model(xb.to(device))
        preds.extend(torch.argmax(out, 1).cpu().numpy())
        labels.extend(yb.numpy())
    return accuracy_score(labels, preds), f1_score(labels, preds, average="weighted")

def train_and_eval(cfg, nested=False):
    """Train one config; early-stop & SELECT on validation F1.
    Logs an MLflow run and returns a summary dict + best checkpoint path."""
    set_seed(SEED)  # same init/shuffling so differences come from the config
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False)

    model = TradingLSTM(
        input_size=len(feature_cols),
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    ).to(device)
    model.lstm.flatten_parameters()

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_val_f1, best_val_acc, best_epoch, no_improve = 0.0, 0.0, 0, 0
    ckpt = f"/tmp/lstm_{cfg['name']}.pt"

    with mlflow.start_run(run_name=cfg["name"], nested=nested):
        mlflow.log_params(cfg)
        mlflow.log_param("num_features", len(feature_cols))
        for epoch in range(cfg["max_epochs"]):
            model.train()
            running = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                running += loss.item() * xb.size(0)
            train_loss = running / len(train_ds)

            val_acc, val_f1 = evaluate(model, val_loader)
            scheduler.step(train_loss)
            mlflow.log_metrics(
                {"train_loss": train_loss, "val_accuracy": val_acc, "val_f1": val_f1},
                step=epoch,
            )

            if val_f1 > best_val_f1:
                best_val_f1, best_val_acc, best_epoch, no_improve = val_f1, val_acc, epoch, 0
                torch.save(model.state_dict(), ckpt)
            else:
                no_improve += 1
                if no_improve >= cfg["patience"]:
                    break
        mlflow.log_metrics(
            {"best_val_f1": best_val_f1, "best_val_accuracy": best_val_acc, "best_epoch": best_epoch}
        )
    print(f"DONE {cfg['name']:14s} | val F1 {best_val_f1:.4f} | val acc {best_val_acc:.4f} | epoch {best_epoch+1}")
    return {"name": cfg["name"], "val_f1": round(best_val_f1, 4), "val_acc": round(best_val_acc, 4),
            "best_epoch": best_epoch + 1, "ckpt": ckpt, "cfg": cfg}

## The Optuna Study

In [0]:
import optuna

results = []
PARENT_RUN_ID = None

def objective(trial):
    cfg = {
        "name": f"optuna_{trial.number:02d}",
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128, 256]),
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "max_epochs": MAX_EPOCHS,
        "patience": EARLY_STOP_PATIENCE,
    }
    res = train_and_eval(cfg, nested=True)
    results.append(res)
    return res["val_f1"]  # Optuna maximizes validation F1

# TPE sampler: learns from past trials to propose better configs (Bayesian).
with mlflow.start_run(run_name="optuna_parent") as parent:
    PARENT_RUN_ID = parent.info.run_id
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS)

print("\nBest validation F1:", round(study.best_value, 4))
print("Best params:", study.best_params)

In [0]:
res_df = pd.DataFrame([
    {"name": r["name"], "val_f1": r["val_f1"], "val_acc": r["val_acc"], "best_epoch": r["best_epoch"],
     **{k: r["cfg"][k] for k in ["hidden_size", "num_layers", "dropout", "lr", "weight_decay", "batch_size"]}}
    for r in results
]).sort_values("val_f1", ascending=False).reset_index(drop=True)
display(res_df)

best = max(results, key=lambda r: r["val_f1"])
print("BEST config (by validation F1):", best["cfg"])

In [0]:
from sklearn.metrics import classification_report

# Rebuild the best model and load the checkpoint that won on validation F1.
best_cfg = best["cfg"]
best_model = TradingLSTM(
    input_size=len(feature_cols),
    hidden_size=best_cfg["hidden_size"],
    num_layers=best_cfg["num_layers"],
    dropout=best_cfg["dropout"],
).to(device)
best_model.load_state_dict(torch.load(best["ckpt"]))
best_model.lstm.flatten_parameters()

test_loader = DataLoader(test_ds, batch_size=best_cfg["batch_size"], shuffle=False)
test_acc, test_f1 = evaluate(best_model, test_loader)
print(f"=== HELD-OUT TEST (best config: {best_cfg['name']}) ===")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test F1:       {test_f1:.4f}\n")

best_model.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.extend(torch.argmax(best_model(xb.to(device)), 1).cpu().numpy())
        labels.extend(yb.numpy())
print(classification_report(labels, preds, target_names=["Down", "Up"]))

In [0]:
from mlflow.models.signature import infer_signature
import joblib

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name=f"best_{best_cfg['name']}_final"):
    mlflow.log_params(best_cfg)
    mlflow.log_metrics({"test_accuracy": test_acc, "test_f1": test_f1})

    example_input = X_test_scaled[:2].astype("float32")  # (2, 30, num_features)
    with torch.no_grad():
        example_output = best_model(torch.from_numpy(example_input).to(device)).cpu().numpy()
    signature = infer_signature(example_input, example_output)

    mlflow.pytorch.log_model(
        best_model,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME,
        signature=signature,
        input_example=example_input,
    )

    joblib.dump(scaler, "/tmp/feature_scaler.pkl")
    mlflow.log_artifact("/tmp/feature_scaler.pkl")
    with open("/tmp/feature_cols.txt", "w") as f:
        f.write("\n".join(feature_cols))
    mlflow.log_artifact("/tmp/feature_cols.txt")
    print("Logged & registered:", REGISTERED_MODEL_NAME)

## Optuna (Conclusion)

15 TPE (Bayesian) trials over the same 3-way time split (train 58,200 / val 3,200 / test 3,650), class-weighted loss, selected on validation F1.

**Winner — `optuna_12`:** hidden=128, layers=2, dropout=0.168, lr=2.06e-4, weight_decay=9.37e-4, batch=256
- Validation F1: **0.4982**
- **Held-out 2025 test: accuracy 0.5005, weighted F1 0.4923**

| Class | Precision | Recall | F1 | Support |
|---|:--:|:--:|:--:|:--:|
| Down | 0.49 | 0.63 | 0.55 | 1787 |
| Up | 0.51 | 0.38 | 0.43 | 1863 |
| **Weighted avg** | 0.50 | 0.50 | **0.49** | 3650 |

**Did Bayesian search beat random search?** No — Optuna's best validation F1 (0.4982) did **not** exceed random search's (0.5168). With only 15 trials and a near-random, very noisy objective (next-day direction), TPE has almost nothing to learn from, so it shows no edge over plain random sampling here. That itself is a finding: smarter search only helps when there is real signal to exploit.

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

CATALOG = "raghavan_trading_signals"
user = spark.sql("SELECT current_user()").collect()[0][0]

# The LSTM tuning experiment (random search + Optuna both log here).
lstm_exp = mlflow.get_experiment_by_name(f"/Users/{user}/raghavan-trading-signals-lstm-tuning")
lstm_runs = mlflow.search_runs(
    experiment_ids=[lstm_exp.experiment_id],
    filter_string="metrics.test_f1 > 0",        # the *_final runs log test_f1
    order_by=["metrics.test_f1 DESC"],
    max_results=1,
)
lstm_run_id = lstm_runs.iloc[0]["run_id"]
lstm_f1 = lstm_runs.iloc[0].get("metrics.test_f1", float("nan"))
lstm_acc = lstm_runs.iloc[0].get("metrics.test_accuracy", float("nan"))
print(f"LSTM    best run {lstm_run_id} | test F1 {lstm_f1:.4f} | test acc {lstm_acc:.4f}")

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

run = mlflow.get_run("a9ff2c64882740d28f884fca68bc9423")
print("run_name :", run.data.tags.get("mlflow.runName"))
print()
print("experiment_id:", run.info.experiment_id)
print()
print("metrics  :", {k: round(v, 4) for k, v in run.data.metrics.items()})
print()
print("params   :", run.data.params)

---